# 0. Setup

In [ ]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

table_name = "working_yearly_with_peers"

# ---

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path)
table = con.table(table_name)

# 2. Execute all counts in a single Ibis/DuckDB aggregation query
agg_result = table.aggregate(
    total_rows=table.count(),
    tfp_nn=table.tfp.count(),
    pc8_nn=table.peer_tfp_pc8.count(),
    pc4_nn=table.peer_tfp_pc4_donut.count(),
    ttwa_nn=table.peer_tfp_ttwa_donut.count(),
).execute()

table_usable = table.drop_null(["tfp", "peer_tfp_pc8", "peer_tfp_pc4_donut", "peer_tfp_ttwa_donut"])

# 3. Extract the computed row
res = agg_result.iloc[0]
total = res['total_rows']

# 4. Construct the stacked display table
df_summary = pd.DataFrame({
    "Filter Name": [
        "Total Rows",
        "TFP",
        "Peer TFP TTWA Donut",
        "Peer TFP PC4 Donut",
        "Peer TFP PC8",
        "Usable rows"
    ],
    "Count": [
        total,
        res['tfp_nn'],
        res['ttwa_nn'],
        res['pc4_nn'],
        res['pc8_nn'],
        table_usable.count().execute()
    ]
})

# 5. Calculate percentage of total rows (formatting as a string with %)
df_summary["Percentage of Total"] = (df_summary["Count"] / total)

# 6. Display the stacked table
display(df_summary.style.format({
    "Count": "{:,}",
    "Percentage of Total": "{:.1%}"
}))

('registered_number', 'year', 'employees', 'fixed_total', 'total_assets', 'average_wage', 'gva1', 'gva2', 'gva1_per_worker', 'gva2_per_worker', 'tfp', 'peer_tfp_pc8', 'peer_tfp_pc4_donut', 'peer_tfp_ttwa_donut')


,Filter Name,Count,Percentage of Total
0,Total Rows,"1,081,520",100.0%
1,TFP,"1,081,520",100.0%
2,Peer TFP TTWA Donut,"1,080,084",99.9%
3,Peer TFP PC4 Donut,"1,070,117",98.9%
4,Peer TFP PC8,"772,842",71.5%
5,Usable rows,"764,537",70.7%


In [ ]:
table_usable

# 1. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$

In [ ]:
from ibis import _
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from utils.f_0_dirs import get_data_dirs

table_working = table_usable
table_results = table_working.select("registered_number", "year")

# 3. Define the 4 model setups to iterate through
# Varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'peer3': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'W_it': [],
        'w_i': [],          # Firm-level controls
        'w_it': [],         # Controls
        'fe': ['i', 't'],
        'description': 'Base: 3 donut peer TFP effects, firm time fixed effects, no controls'
    },
    'peer3_no_firm_fe': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'W_it': [],
        'w_i': [],
        'w_it': [],
        'fe': ['t'],
        'description': 'Base - firm FE',
        'include': False
    },
    'peer3_employees': {
        'Y': 'tfp',
        'X': ['peer_tfp_ttwa_donut', 'peer_tfp_pc4_donut', 'peer_tfp_pc8'],
        'W_it': ['employees'],
        'w_i': [],
        'w_it': [],
        'fe': ['i', 't'],
        'description': 'Base + employees control',
        'include': False
    }
}

# Dictionary to store the parameter outputs (\beta_K, \beta_L) and model summaries
parameter_tables = {}
output_str = ""
model_count = 0
for name, mod in models.items():
    if not mod.get('include', True):
        continue

    # Mutate to dynamically log-transform all columns, adding ln_ prefix to column name
    log_keys = ['W_it']
    lvl_keys = ['Y', 'X', 'w_i', 'w_it']
    log_raw_params: list[str] = []
    lvl_raw_params: list[str] = []
    for key in log_keys:
        if isinstance(mod[key], list):
            log_raw_params.extend(mod[key])
        else:
            log_raw_params.append(mod[key])
    for key in lvl_keys:
        if isinstance(mod[key], list):
            lvl_raw_params.extend(mod[key])
        else:
            lvl_raw_params.append(mod[key])

    full_params = (
        ['registered_number', 'year'] +
        [f'ln_{p}' for p in log_raw_params] +
        [f'{p}' for p in lvl_raw_params]
    )
    if len(log_raw_params) > 0:
        table_logged = (
            table_working
            .filter(_[p] > 0 for p in log_raw_params)
            .mutate(**{f'{p}': np.log(_[p]) for p in log_raw_params})
            .rename({ f'ln_{p}': f'{p}' for p in log_raw_params })
        )
    else:
        table_logged = table_working
    table_logged = table_logged.select(full_params)
    if 'Y' in log_keys:
        table_start = table_logged.rename({ 'ln_Y': f'ln_{mod["Y"]}' })
    else:
        table_start = table_logged.rename({ 'Y': f'{mod["Y"]}' })
    table_start = table_start.execute()

    # 1. Execute into a Pandas DataFrame and set the MultiIndex for linearmodels
    df_model = table_start.set_index(['registered_number', 'year'])
    regressor_params = [p for p in full_params if p not in ['registered_number', 'year', f'ln_{mod["Y"]}', mod["Y"]]]

    # Define Endogenous (Y) and Exogenous (X) variables
    Y = df_model['ln_Y'] if 'ln_Y' in df_model.columns else df_model['Y']
    X = sm.add_constant(df_model[list(regressor_params)])
    
    # 2. Estimate the model with Firm and Year Fixed Effects
    mod_ols = PanelOLS(Y, X,
                       entity_effects='i' in mod['fe'],
                       time_effects='t' in mod['fe']
                    )
    
    # Fit model with firm-clustered standard errors
    res = mod_ols.fit(cov_type='clustered', cluster_entity=True)
    
    # Extract \beta results iterating through full_params
    beta = { p: res.params[p] for p in regressor_params }
    
    # 5. Store the results and parameters
    parameter_tables[name] = {
        **beta,
        # Safely extract time effects if they exist
        'gamma_t': res.estimated_effects.xs('time_effects', level=1) if 'time_effects' in res.estimated_effects.index.names else None, 
        'summary': res.summary
    }
    print(f"✅ Model '{name}' estimated: {", ".join([f'{k}={v:.3f}' for k, v in beta.items()])}")
    print(f"Model summary:\n{res.summary}")
    output_str += f"Model '{name}': {mod['description']}\n"
    output_str += f"{res.summary}\n\n"
    output_str += "=".format(87) + "\n\n"
    model_count += 1

print(f"Panel regressions complete. {model_count} models, writing.")
with open(dirs.output_dir / "results_1_llm.txt", "w") as f:
    f.write(output_str)

✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Model summary:
                          PanelOLS Estimation Summary                           
Dep. Variable:                      Y   R-squared:                        0.0388
Estimator:                   PanelOLS   R-squared (Between):              0.1089
No. Observations:              764537   R-squared (Within):               0.0396
Date:                Sat, Aug 22 2026   R-squared (Overall):              0.1285
Time:                        13:33:09   Log-likelihood                -3.354e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      8843.1
Entities:                      107143   P-value                           0.0000
Avg Obs:                       7.1357   Distribution:                F(3,657373)
Min Obs:                       1.0000                                       

# 2. Industries group model
$$y_{it} = \alpha_i + \gamma_t + x_{it}\theta + \sum_{k=1}^4 \beta_k E[TFP\_s_{-i, g_k, t}] + \delta_1 E[TFP\_s_{-i, SIC2, t}] + \delta_2 E[TFP\_s_{-i, SIC6, t}] + \epsilon_{it}$$